# Jupyter Notebook to help manage calculations

In [32]:
import os
import subprocess
import sys
import arkane

DFT_DIR = os.path.join(os.environ['AUTOSCIENCE_REPO'], 'dft')
sys.path.append(DFT_DIR)
import autotst_wrapper
import autotst.reaction

sys.path.append(os.environ['DATABASE_DIR'])
import database_fun
import rmgpy.chemkin

import job_manager

import importlib
importlib.reload(autotst_wrapper)

<module 'autotst_wrapper' from '/work/westgroup/harris.se/autoscience/reaction_calculator/dft/autotst_wrapper.py'>

In [5]:
def get_reaction_species(reaction):
    if type(reaction) == int:
        reaction = database_fun.index2reaction(reaction)
        
    reaction_string = database_fun.get_unique_string(reaction)
    tokens = reaction_string.split('=')
    reactants = [int(i) for i in tokens[0].split('+')]
    products = [int(i) for i in tokens[-1].split('+')]
    return reactants + products

In [6]:
get_reaction_species(10106)

[18, 860, 11, 863]

# What do we need to calculate?

In [7]:
# # DIB-min-1 most sensitive
# species_to_calculate = [599, 15, 21, 185, 649, 636, 607, 634, 131, 598, 600, 140, 604, 601, 613, 88]
# reactions_to_calculate = [9358, 9060, 9769, 9319, 9641, ]
# for r in reactions_to_calculate:
#     species_to_calculate += get_reaction_species(r)
# species_to_calculate = sorted(list(set(species_to_calculate)))
# reactions_to_calculate = sorted(reactions_to_calculate)

# # DIB-MAX-1 most sensitive
# species_to_calculate = [598, 857, 856, 858, 599, 602, 601, 605, 604, 624]
# reactions_to_calculate = [9177, 10224, 10287, 9224, 10314, 10222]
# for r in reactions_to_calculate:
#     species_to_calculate += get_reaction_species(r)
# species_to_calculate = sorted(list(set(species_to_calculate)))
# reactions_to_calculate = sorted(reactions_to_calculate)



# # Actual DIB-min-1
# species_to_calculate = [21, 607, 634, 599, 636, 649, 15, 185]
# # reactions_to_calculate = [9064, 9018, 9060, 9358]
# reactions_to_calculate = [9060, 9358]
# for r in reactions_to_calculate:
#     species_to_calculate += get_reaction_species(r)
# species_to_calculate = sorted(list(set(species_to_calculate)))
# reactions_to_calculate = sorted(reactions_to_calculate)


# Actual DIB-MAX-1
species_to_calculate = []
reactions_to_calculate = [10106, 10105, 10351, 10168, 10371, 10123, 10167, 10102, 10155, 10111]
for r in reactions_to_calculate:
    species_to_calculate += get_reaction_species(r)
species_to_calculate = sorted(list(set(species_to_calculate)))
reactions_to_calculate = sorted(reactions_to_calculate)




# # Butane RMG-MAX-1 Round 1
# species_to_calculate = [87, 84, 85, 90, 88]
# reactions_to_calculate = [213, 324, 714, 1111, 804]
# assert len(species_to_calculate) + len(reactions_to_calculate) == 10
# for r in reactions_to_calculate:
#     species_to_calculate += get_reaction_species(r)
# species_to_calculate = sorted(list(set(species_to_calculate)))
# reactions_to_calculate = sorted(reactions_to_calculate)

# # Butane RMG-min-1 Round 1
# species_to_calculate = [4, 57, 280, 60, 61, 73, 21, 72, 32]
# reactions_to_calculate = [4721]
# assert len(species_to_calculate) + len(reactions_to_calculate) == 10
# for r in reactions_to_calculate:
#     species_to_calculate += get_reaction_species(r)
# species_to_calculate = sorted(list(set(species_to_calculate)))
# reactions_to_calculate = sorted(reactions_to_calculate)


# reactions_to_calculate = [
#     288, 4724, 5046, 4778, 4736, 4729, 4728, 5047,
#     4779, 286, 246, 5596, 808, 915, 4737, 5446,
#     324, 4738, 7841, 804, 809, 4721, 945, 213, 289,
#     422, 805, 4796, 1077, 1111, 1706, 4917, 417, 319,
#     313, 278, 314, 52, 5056, 405, 5102, 521, 404, 410,
#     4733, 296, 321, 301, 280, 253, 459, 1736, 1778
# ]
# species_to_calculate = [4, 57, 280, 60, 61, 73, 21, 72, 32, 48, 15, 70, 85, 62, 275, 294, 38, 31, 25, 36]
# for r in reactions_to_calculate:
#     species_to_calculate += get_reaction_species(r)
# species_to_calculate = sorted(list(set(species_to_calculate)))
# reactions_to_calculate = sorted(reactions_to_calculate)


# reactions_to_calculate = [
#     288, 4724, 5046, 4778, 4736, 4729, 4728, 50, 4752, 5047,
#     4779, 286, 246, 5596, 808, 915, 4732, 518, 4737, 5446,
#     324, 4738, 7841, 804, 809, 4721, 245, 945, 213, 289,
#     422, 805, 4796, 1077, 1111, 1706, 4917, 417, 319,
#     313, 278, 314, 52, 5056, 405, 5102, 521, 404, 410,
#     4733, 296, 321, 301, 280, 253, 459, 1736, 1778, 299
# ]

# problem_reactions = [50, 4752, 4732, 518, 245, 299]


In [8]:
species_to_calculate

[11,
 13,
 16,
 17,
 18,
 19,
 20,
 21,
 26,
 612,
 625,
 626,
 855,
 856,
 857,
 859,
 860,
 862,
 863]

# Check which things are complete

In [9]:
def run_command(command):
    p = subprocess.Popen(
        command,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    output = p.communicate()
    text = output[0].decode('utf-8')
    text_list = text.split('\n')
    if text_list[-1] == '':
        text_list = text_list[:-1]
    return text_list

In [10]:
# make sure 'dlpno' AND 'orca terminated normally' are in the logfile that was used
DFT_DIR = os.environ['DFT_DIR']

finished_sp_single_points = run_command('grep -il "orca terminated normally" '+ os.path.join(DFT_DIR, 'thermo/species_*/single_point/conformer.out'))
finished_ts_single_points = run_command('grep -il "orca terminated normally" '+ os.path.join(DFT_DIR, 'kinetics/reaction_*/single_point/conformer.out'))


finished_arkane_thermo = run_command('grep -il "dlpno" '+ os.path.join(DFT_DIR, 'thermo/species_*/arkane/RMG_libraries/thermo.py'))
finished_arkane_kinetics = run_command('grep -il "dlpno" '+ os.path.join(DFT_DIR, 'kinetics/reaction_*/arkane/RMG_libraries/reactions.py'))

# Go through species

In [11]:
unfinished_species = []
finished_species = []
for i in species_to_calculate:
    orca_logname = os.path.join(DFT_DIR, f'thermo/species_{i:04}/single_point/conformer.out')
    arkane_fname = os.path.join(DFT_DIR, f'thermo/species_{i:04}/arkane/RMG_libraries/thermo.py')
    if orca_logname not in finished_sp_single_points or arkane_fname not in finished_arkane_thermo:
        print(f'Species {i} not done yet')
        unfinished_species.append(i)
    else:
        finished_species.append(i)
if len(unfinished_species) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_species)} left to calculate')

Species 626 not done yet
Species 863 not done yet

2 left to calculate


In [12]:
finished_species

[11, 13, 16, 17, 18, 19, 20, 21, 26, 612, 625, 855, 856, 857, 859, 860, 862]

# Go through reactions

In [13]:
unfinished_reactions = []
for i in reactions_to_calculate:
    orca_logname = os.path.join(DFT_DIR, f'kinetics/reaction_{i:06}/single_point/conformer.out')
    arkane_fname = os.path.join(DFT_DIR, f'kinetics/reaction_{i:06}/arkane/RMG_libraries/reactions.py')
    if orca_logname not in finished_ts_single_points or arkane_fname not in finished_arkane_kinetics:
        print(f'Reaction {i} not done yet')
        unfinished_reactions.append(i)

if len(unfinished_reactions) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_reactions)} left to calculate')

Reaction 10102 not done yet
Reaction 10106 not done yet
Reaction 10111 not done yet
Reaction 10123 not done yet
Reaction 10155 not done yet
Reaction 10167 not done yet
Reaction 10168 not done yet
Reaction 10351 not done yet
Reaction 10371 not done yet

9 left to calculate


# Species

## 1. Geometry Optimization

In [14]:
unfinished_opt = []
for i in unfinished_species:
    conformer_dir = os.path.join(DFT_DIR, 'thermo', f'species_{i:04}', 'conformers')
    geo_opt_completed = autotst_wrapper.conformers_done_optimizing(conformer_dir, completion_threshold=0.5)
    if not geo_opt_completed:
        print(f'Species {i} optimization not done yet')
        unfinished_opt.append(i)

if len(unfinished_opt) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_opt)} left to calculate')

DONE!


In [15]:
unfinished_opt


[]

In [ ]:
# option to set off species optimizations
# for i in [627, 629, 697]:
for i in unfinished_opt:
    print(f'Starting species conformer optimization {i}')
    autotst_wrapper.screen_species_conformers(i)
    autotst_wrapper.optimize_conformers(i)

## 2. Rotor scans

In [16]:
# Using fixed point scans now
def get_n_sp_rotors(species_index):
    species_dir = os.path.join(DFT_DIR, 'thermo', f'species_{species_index:04}')
    conformer_dir = os.path.join(species_dir, 'conformers')

    direction = 'forward'
    rmg_species = database_fun.index2species(species_index)
    smiles = rmg_species.smiles
    
    # Get the lowest energy conformer from the overall result -- look in the arkane folder
    conformer_file = autotst_wrapper.get_lowest_valid_conformer(conformer_dir, species_index)
    
    new_cf = autotst.species.Conformer(smiles=smiles)  # TODO make this from adjacency list?
    new_cf._ase_molecule = autotst_wrapper.get_gaussian_file_geometry(conformer_file)
    new_cf.update_coords_from(mol_type="ase")
    torsions = new_cf.get_torsions()  # TODO - is this only the nonterminal ones?

    # get the rotors
    return len(torsions)

In [25]:
unfinished_rotors = []
for i in unfinished_species:
    rotor_dir = os.path.join(DFT_DIR, 'thermo', f'species_{i:04}', 'rotors')
    n_rotors = get_n_sp_rotors(i)
    for j in range(n_rotors):
        if not autotst_wrapper.species_rotor_complete(i, j):
            print(f'Species {i} rotor {j} not done yet')
            unfinished_rotors.append(i)
    print()
        
if len(unfinished_rotors) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_rotors)} left to calculate')

2025-01-10 09:10:56.255321 Lowest energy conformer is /work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_0626/conformers/conformer_0002.log
/work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_0626/rotors/rotor_0001.log has errors
Species 626 rotor 1 not done yet

2025-01-10 09:10:57.153349 Lowest energy conformer is /work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_0863/conformers/conformer_0000.log
Species 863 rotor 0 not done yet


2 left to calculate


In [17]:
unfinished_rotors = []
for i in unfinished_species:
    rotor_dir = os.path.join(DFT_DIR, 'thermo', f'species_{i:04}', 'rotors')
    rotors = not os.path.exists(os.path.join(rotor_dir, 'NO_ROTORS.txt'))
    if rotors:
        rotors_completed = autotst_wrapper.conformers_done_optimizing(rotor_dir, completion_threshold=1.0, base_name='rotor_')
        if not rotors_completed:
            print(f'Species {i} rotors not done yet')
            unfinished_rotors.append(i)
            print()
        
if len(unfinished_rotors) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_rotors)} left to calculate')

Not done optimizing:
0.8571428571428571
6 finished
6 good
1 incomplete
3 unlisted
Species 626 rotors not done yet

Not done optimizing:
0.8333333333333334
5 finished
5 good
0 incomplete
5 unlisted
Species 863 rotors not done yet


2 left to calculate


In [18]:
# see if the fixed rotors are missing?
for species_index in unfinished_rotors:
    # print out which rotors are unfinished
    rotor_dir = os.path.join(DFT_DIR, 'thermo', f'species_{species_index:04}', 'rotors')
    # Using grep instead of arkane.ess.gaussian because I trust it more not to interfere with currently running jobs
    finished_rotors = run_command('grep -il "Normal termination of Gaussian" '+ os.path.join(rotor_dir, 'rotor_*.log'))

    n_rotors = get_n_sp_rotors(species_index)
    print(f'For species {species_index}, {n_rotors} rotors to calculate:')
    
    for rotor_index in range(n_rotors):
        rotor_logfile = os.path.join(rotor_dir, f'rotor_{rotor_index:04}.log')
        if not rotor_logfile in finished_rotors:
            print(f'\tRotor {rotor_index} incomplete')
            # check out the fixed rotor options
            fixed_rotor_dir = os.path.join(DFT_DIR, 'thermo', f'species_{species_index:04}', 'rigid_rotors')
            scan_energy_file = os.path.join(fixed_rotor_dir, f'rotor_{rotor_index:04}_scan_energies.txt')
            if not os.path.exists(scan_energy_file):
                try:
                    autotst_wrapper.assemble_rotor_scan_energies(fixed_rotor_dir, rotor_index)
                except ValueError:
                    print(f'\tNot enough complete scans for rotor {rotor_index}')
            if not os.path.exists(scan_energy_file):
                print(f'\tNo fixed scans for rotor {rotor_index}')
            else:
                print(f'\tFixed scans complete for rotor {rotor_index}')
    print()
    
    
    
#     fixed_rotor_dir = os.path.join(DFT_DIR, 'thermo', f'species_{i:04}', 'rigid_rotors')

2025-01-10 08:28:33.317680 Lowest energy conformer is /work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_0626/conformers/conformer_0002.log
For species 626, 7 rotors to calculate:
	Rotor 1 incomplete
last rotorfile /work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_0626/rigid_rotors/rotor_0001_0020.log
	Not enough complete scans for rotor 1
	No fixed scans for rotor 1

2025-01-10 08:28:33.826902 Lowest energy conformer is /work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_0863/conformers/conformer_0000.log
For species 863, 6 rotors to calculate:
	Rotor 0 incomplete
last rotorfile /work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_0863/rigid_rotors/rotor_0000_0020.log
	Not enough complete scans for rotor 0
	No fixed scans for rotor 0



In [26]:
# see how many jobs are currently running
jobs = run_command('squeue -u harris.se')
print(f'{len(jobs) - 1} jobs running')
print()
for line in jobs:
    print(line)


32 jobs running

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
          46101884       gpu sys/dash harris.s  R      43:44      1 c2175
        46101909_9     short parallel harris.s  R      38:17      1 d0107
       46101909_10     short parallel harris.s  R      38:17      1 d0110
       46101909_11     short parallel harris.s  R      38:17      1 d0127
       46101909_12     short parallel harris.s  R      38:17      1 d0118
       46101909_13     short parallel harris.s  R      38:17      1 d0118
       46101909_14     short parallel harris.s  R      38:17      1 d0095
       46101909_15     short parallel harris.s  R      38:17      1 d0095
       46101909_16     short parallel harris.s  R      38:17      1 d0095
       46101909_17     short parallel harris.s  R      38:17      1 d0097
       46101909_18     short parallel harris.s  R      38:17      1 d0097
       46101909_19     short parallel harris.s  R      38:17      1 d0097
       461

In [ ]:
# start another job

species_index = 863
rotor_index = 0

species_dir = os.path.join(DFT_DIR, 'thermo', f'species_{species_index:04}')

# submit the job
slurm_run_file = os.path.join(DFT_DIR, 'parallel_run_rigid_rotor.sh')
start_dir = os.getcwd()
os.chdir(species_dir)
fixed_sp_rotor_job = job_manager.SlurmJob()
slurm_cmd = f'sbatch {slurm_run_file} {species_index} {rotor_index}'
fixed_sp_rotor_job.submit(slurm_cmd)
os.chdir(start_dir)

In [22]:
# assemble rotors
species_index = 863
rotor_index = 0


# rotor_dir = '/work/westgroup/SCRATCH/sevy_calcs/reaction_010105/rotors/rotor_0006'
rotor_dir = os.path.join(DFT_DIR, 'thermo', f'species_{species_index:04}', 'rigid_rotors')
autotst_wrapper.assemble_rotor_scan_energies(rotor_dir, rotor_index)


last rotorfile /work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_0863/rigid_rotors/rotor_0000_0020.log


ValueError: missing rotor TS scan energies for rotor 0:04

In [ ]:
# TODO - find the partially completed rotors and run those as fixed scans

In [ ]:
# option to set off species
for i in unfinished_rotors:
    print(f'Starting species rotor scans {i}')
    autotst_wrapper.setup_rotors(i)
    autotst_wrapper.run_rotors(i)

## 3. DLPNO-CCSDT(T)-F12 Single Point Calculations

In [27]:
unfinished_single_points = []
for i in unfinished_species:
    orca_logname = os.path.join(DFT_DIR, f'thermo/species_{i:04}/single_point/conformer.out')
    if orca_logname not in finished_sp_single_points:
        print(f'Species {i} single point calculation not done yet')
        unfinished_single_points.append(i)
        print()
        
if len(unfinished_single_points) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_single_points)} left to calculate')

DONE!


In [ ]:
unfinished_single_points

In [ ]:
# TODO run single points
# subset = [unfinished_single_points[0]]
# assert len(subset) <= 5
for i in [295, 296, 297]:
    print(f'Starting species single point calc {i}')
    autotst_wrapper.setup_single_point(i, calc_type='species', force_rerun=True, parallel=True)
#     autotst_wrapper.run_single_point(i, calc_type='species', force_rerun=True)


## 4. Run Arkane

In [28]:
unfinished_species_arkanes = []
for i in unfinished_species:
    arkane_result = os.path.join(DFT_DIR, f'thermo/species_{i:04}/arkane/RMG_libraries/thermo.py')
    if arkane_result not in finished_arkane_thermo:
        print(f'Species {i} Arkane not done yet')
        unfinished_species_arkanes.append(i)
        print()
        
if len(unfinished_species_arkanes) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_species_arkanes)} left to calculate')

Species 626 Arkane not done yet

Species 863 Arkane not done yet


2 left to calculate


In [ ]:
# option to set off species optimizations
# subset = unfinished_species_arkanes
# assert len(subset) < 5
# for i in [634, 636, 649]:
for i in [625]:
    print(f'Starting species arkane run {i}')
    autotst_wrapper.setup_arkane_species(i, force_rerun=True)
    autotst_wrapper.run_arkane_species(i, force_rerun=True)


# Reaction Calculation Steps

## 1. Shell Optimization (reaction center frozen)

In [33]:
unfinished_shell = []
finished_shell = []
for i in unfinished_reactions:
    conformer_dir = os.path.join(DFT_DIR, 'kinetics', f'reaction_{i:06}', 'shell')
    geo_opt_completed = autotst_wrapper.conformers_done_optimizing(conformer_dir, completion_threshold=0.3, base_name='fwd_ts_')
    if not geo_opt_completed:
        print(f'Reaction {i} shell optimization not done yet')
        unfinished_shell.append(i)
    else:
        finished_shell.append(i)

if len(unfinished_shell) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_shell)} left to calculate')

Not done optimizing:
0.0
0 finished
0 good
10 incomplete
0 unlisted
Reaction 10123 shell optimization not done yet

1 left to calculate


In [ ]:
for i in unfinished_shell:
    print(f'Starting reaction shell optimization {i}')
    autotst_wrapper.setup_opt(i, 'shell')
    autotst_wrapper.run_opt(i, 'shell')

In [34]:
finished_shell

[10102, 10106, 10111, 10155, 10167, 10168, 10351, 10371]

## 2. Center Optimization (shell frozen - optimize reaction center to loose TS)

In [36]:
unfinished_center = []
finished_center = []
for i in finished_shell:
    conformer_dir = os.path.join(DFT_DIR, 'kinetics', f'reaction_{i:06}', 'center')
    geo_opt_completed = autotst_wrapper.conformers_done_optimizing(conformer_dir, completion_threshold=0.3, base_name='fwd_ts_')
    if not geo_opt_completed:
        print(f'Reaction {i} center optimization not done yet')
        unfinished_center.append(i)
    else:
        finished_center.append(i)
        

if len(unfinished_center) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_center)} left to calculate')

Not done optimizing:
0.0
0 finished
0 good
1 incomplete
9 unlisted
Reaction 10111 center optimization not done yet
Not done optimizing:
0.0
0 finished
0 good
4 incomplete
6 unlisted
Reaction 10155 center optimization not done yet

2 left to calculate


In [37]:
finished_center

[10102, 10106, 10167, 10168, 10351, 10371]

In [ ]:
for i in unfinished_center:
    print(f'Starting reaction center optimization {i}')
    autotst_wrapper.setup_opt(i, 'center')
    autotst_wrapper.run_opt(i, 'center')

## 3. Overall TS Optimization (no constraints)

In [48]:
unfinished_overall = []
finished_overall = []
for i in finished_center:
    conformer_dir = os.path.join(DFT_DIR, 'kinetics', f'reaction_{i:06}', 'overall')
    geo_opt_completed = autotst_wrapper.conformers_done_optimizing(conformer_dir, completion_threshold=0.01, base_name='fwd_ts_')
    if not geo_opt_completed:
        print(f'Reaction {i} overall optimization not done yet')
        unfinished_overall.append(i)
    else:
        finished_overall.append(i)

if len(unfinished_overall) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_overall)} left to calculate')

Not done optimizing:
1.0
1 finished
0 good
0 incomplete
9 unlisted
Reaction 10351 overall optimization not done yet
No conformers with glob string /work/westgroup/harris.se/autoscience/reaction_calculator/dft/kinetics/reaction_010371/overall/fwd_ts_*.com
Reaction 10371 overall optimization not done yet

2 left to calculate


In [49]:
finished_overall

[10102, 10106, 10167, 10168]

In [ ]:
for i in unfinished_overall:
    print(f'Starting reaction overall optimization {i}')
    autotst_wrapper.setup_opt(i, 'overall')
    autotst_wrapper.run_opt(i, 'overall')

## 4. Reaction TS Rotor Scans

In [50]:
# Using fixed point scans now
def get_n_rotors(reaction_index):
    reaction_dir = os.path.join(DFT_DIR, 'kinetics', f'reaction_{reaction_index:06}')
    overall_dir = os.path.join(reaction_dir, 'overall')

    direction = 'forward'
    rmg_reaction = database_fun.index2reaction(reaction_index)
    reaction_smiles = database_fun.reaction_index2smiles(reaction_index)
    reaction = autotst.reaction.Reaction(label=reaction_smiles)  # going back to this even though it's not dependable

    
    # Get the lowest energy conformer from the overall result -- look in the arkane folder
    conformer_file = autotst_wrapper.get_lowest_energy_gaussian_file(overall_dir)

    reaction.ts[direction][0]._ase_molecule = autotst_wrapper.get_gaussian_file_geometry(conformer_file)
    reaction.ts[direction][0].update_coords_from(mol_type="ase")
    torsions = reaction.ts[direction][0].get_torsions()
        
    # get the rotors
    return len(reaction.ts[direction][0].torsions)

In [51]:
unfinished_rotors = []
finished_rotors = []
for i in finished_overall:
    rotor_dir = os.path.join(DFT_DIR, 'thermo', f'species_{i:04}', 'rotors')
    n_rotors = get_n_rotors(i)
    has_incomplete = False
    for j in range(n_rotors):
        if not autotst_wrapper.reaction_rotor_complete(i, j):
            print(f'Reaction {i} rotor {j} not done yet')
            has_incomplete = True
    if has_incomplete:
        unfinished_rotors.append(i)
    else:
        finished_rotors.append(i)
    print()
        
if len(unfinished_rotors) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_rotors)} left to calculate')

RDKit WARNING: [09:29:57] WARNING: not removing hydrogen atom without neighbors
[09:29:57] WARNING: not removing hydrogen atom without neighbors
RDKit WARNING: [09:29:57] WARNING: not removing hydrogen atom without neighbors
RDKit WARNING: [09:29:57] WARNING: not removing hydrogen atom without neighbors
[09:29:57] WARNING: not removing hydrogen atom without neighbors
[09:29:57] WARNING: not removing hydrogen atom without neighbors
RDKit WARNING: [09:29:57] WARNING: not removing hydrogen atom without neighbors
[09:29:57] WARNING: not removing hydrogen atom without neighbors


Reaction 10102 rotor 1 not done yet
Reaction 10102 rotor 2 not done yet
Reaction 10102 rotor 3 not done yet
Reaction 10102 rotor 4 not done yet
Reaction 10102 rotor 5 not done yet

Reaction 10106 rotor 0 not done yet
Reaction 10106 rotor 1 not done yet
Reaction 10106 rotor 2 not done yet
Reaction 10106 rotor 3 not done yet
Reaction 10106 rotor 4 not done yet
Reaction 10106 rotor 5 not done yet
Reaction 10106 rotor 6 not done yet
Reaction 10106 rotor 7 not done yet

Reaction 10167 rotor 0 not done yet
Reaction 10167 rotor 1 not done yet
Reaction 10167 rotor 2 not done yet
Reaction 10167 rotor 3 not done yet
Reaction 10167 rotor 4 not done yet
Reaction 10167 rotor 5 not done yet
Reaction 10167 rotor 6 not done yet
Reaction 10167 rotor 7 not done yet

Reaction 10168 rotor 4 not done yet
Reaction 10168 rotor 5 not done yet
Reaction 10168 rotor 6 not done yet
Reaction 10168 rotor 7 not done yet


4 left to calculate


In [ ]:
get_n_rotors(10105)

In [ ]:
overall_dir = '/work/westgroup/harris.se/autoscience/reaction_calculator/dft/kinetics/reaction_010105/overall'

In [ ]:
autotst_wrapper.get_lowest_energy_gaussian_file(overall_dir)

In [ ]:
autotst_wrapper.get_lowest_valid_ts(overall_dir)

In [ ]:
get_n_rotors(9358)

In [ ]:
unfinished_ts_rotors = []
for i in unfinished_reactions:
    
    
    
    
    
    conformer_dir = os.path.join(DFT_DIR, 'kinetics', f'reaction_{i:06}', 'rigid_rotors')
    geo_opt_completed = autotst_wrapper.conformers_done_optimizing(conformer_dir, completion_threshold=0.9, base_name='rotor_')
    if not geo_opt_completed:
        print(f'Reaction {i} TS rotors not done yet')
        unfinished_ts_rotors.append(i)

if len(unfinished_ts_rotors) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_ts_rotors)} left to calculate')

In [ ]:
subset = unfinished_ts_rotors[1:5]

In [ ]:
subset

In [ ]:
subset = unfinished_ts_rotors[0:5]
assert len(subset) <= 5

for i in subset:
    print(f'Starting reaction TS rotors {i}')
    autotst_wrapper.setup_ts_rotors(i, force_rerun=True)
    autotst_wrapper.run_ts_rotors(i, force_rerun=True)

## 5. Reaction Single Point Calculations

In [ ]:
unfinished_rxn_single_points = []
for i in unfinished_reactions:
    orca_logname = os.path.join(DFT_DIR, f'kinetics/reaction_{i:06}/single_point/conformer.out')
    if orca_logname not in finished_ts_single_points:
        print(f'Reaction {i} single point not done yet')
        unfinished_rxn_single_points.append(i)

if len(unfinished_rxn_single_points) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_rxn_single_points)} left to calculate')

In [ ]:
print(database_fun.index2reaction(804))

In [ ]:
for i in [804]:
    print(f'Starting reaction single point {i}')
    autotst_wrapper.setup_single_point(i, calc_type='reaction', force_rerun=True, parallel=True)
    autotst_wrapper.run_single_point(i, calc_type='reaction', force_rerun=True)

#     try:
#         autotst_wrapper.setup_single_point(i, calc_type='reaction', force_rerun=True, parallel=False)
#         autotst_wrapper.run_single_point(i, calc_type='reaction', force_rerun=True)
#     except (TypeError, arkane.exceptions.LogError):
#         pass

## 6. Reaction Arkane

In [ ]:
unfinished_rxn_arkane = []
for i in unfinished_reactions:
    arkane_result = os.path.join(DFT_DIR, f'kinetics/reaction_{i:06}/arkane/RMG_libraries/reactions.py')
    if arkane_result not in finished_arkane_kinetics:
        print(f'Reaction {i} Arkane not done yet')
        unfinished_rxn_arkane.append(i)

if len(unfinished_rxn_arkane) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_rxn_arkane)} left to calculate')

In [ ]:
for i in [9060]:
    print(f'Starting reaction Arkane {i}')
    autotst_wrapper.setup_arkane_reaction(i, force_rerun=True)
    autotst_wrapper.run_arkane_reaction(i, force_rerun=True)